In [1]:
import pandas as pd
import tensorflow as tf

# 1. Load Data
df = pd.read_csv(
    "https://drive.google.com/uc?id=1AZRfFoyekqSYpri5183RmJjciRGz_ood",
    sep=",",
    index_col="datetime",
    header=0,
)

# 2. Normalisasi (Min-Max Scaling agar data berada di rentang 0-1)
def normalize_series(data, min_val, max_val):
    data = (data - min_val) / (max_val - min_val)
    return data

data = df.values
data = normalize_series(data, data.min(axis=0), data.max(axis=0))

N_FEATURES = len(df.columns)

# 3. Split Data (50% train, 50% valid)
SPLIT_TIME = int(len(data) * 0.5)
x_train = data[:SPLIT_TIME]
x_valid = data[SPLIT_TIME:]

# 4. Fungsi Windowing (Membentuk input dan target)
def windowed_dataset(series, batch_size, n_past=24, n_future=24, shift=1):
    ds = tf.data.Dataset.from_tensor_slices(series)
    # Membuat window geser: ukuran total (past + future)
    ds = ds.window(size=n_past + n_future, shift=shift, drop_remainder=True)
    ds = ds.flat_map(lambda w: w.batch(n_past + n_future))
    # Memisahkan n_past sebagai input X, dan n_future sebagai target Y
    ds = ds.map(lambda w: (w[:n_past], w[n_past:]))
    return ds.batch(batch_size).prefetch(1)

BATCH_SIZE = 32
N_PAST = 24
N_FUTURE = 24
SHIFT = 1

train_set = windowed_dataset(x_train, BATCH_SIZE, N_PAST, N_FUTURE, SHIFT)
valid_set = windowed_dataset(x_valid, BATCH_SIZE, N_PAST, N_FUTURE, SHIFT)

# 5. Arsitektur Model (Dense/MLP)
model = tf.keras.models.Sequential([
    # Input shape: (24, N_FEATURES), Dense akan mem-flatten data ini
    tf.keras.layers.Flatten(input_shape=(N_PAST, N_FEATURES)), 
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    # Output layer harus menghasilkan n_future * N_FEATURES
    tf.keras.layers.Dense(N_FUTURE * N_FEATURES),
    tf.keras.layers.Reshape([N_FUTURE, N_FEATURES]) # Mengembalikan bentuk ke (24, N_FEATURES)
])

# 6. Callback untuk efisiensi training
class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if (logs.get('mae') < 0.055 and logs.get('val_mae') < 0.055):
            print("\nMAE target tercapai, menghentikan training!")
            self.model.stop_training = True

callbacks = myCallback()

# 7. Kompilasi dan Training
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss='mae', optimizer=optimizer, metrics=["mae"])

model.fit(train_set, validation_data=valid_set, epochs=100, callbacks=callbacks)

train_pred = model.predict(train_set)
print (train_pred[0][0])

Epoch 1/100


c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


   1345/Unknown 17s 10ms/step - loss: 0.1100 - mae: 0.1100

c:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1349/1349 ━━━━━━━━━━━━━━━━━━━━ 29s 19ms/step - loss: 0.0826 - mae: 0.0826 - val_loss: 0.0651 - val_mae: 0.0651
Epoch 2/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 23s 17ms/step - loss: 0.0620 - mae: 0.0620 - val_loss: 0.0616 - val_mae: 0.0616
Epoch 3/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 15s 11ms/step - loss: 0.0602 - mae: 0.0602 - val_loss: 0.0599 - val_mae: 0.0599
Epoch 4/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 25s 19ms/step - loss: 0.0586 - mae: 0.0586 - val_loss: 0.0570 - val_mae: 0.0570
Epoch 5/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 31s 23ms/step - loss: 0.0579 - mae: 0.0579 - val_loss: 0.0564 - val_mae: 0.0564
Epoch 6/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 24s 18ms/step - loss: 0.0573 - mae: 0.0573 - val_loss: 0.0575 - val_mae: 0.0575
Epoch 7/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 23s 17ms/step - loss: 0.0556 - mae: 0.0556 - val_loss: 0.0559 - val_mae: 0.0559
Epoch 8/100
1349/1349 ━━━━━━━━━━━━━━━━━━━━ 59s 44ms/step - loss: 0.0555 - mae: 0.0555 - val_loss: 0.0547 - val_mae: 0.0547
Epoch 9/100
1346/1349 ━━━━━━

In [2]:
model.save("model.h5")